# Ноутбук 5. Uplift-модели: сравнение подходов к оценке индивидуального эффекта коммуникации

## О чём этот ноутбук

В предыдущих ноутбуках построены две **классические модели кредитного скоринга**:
логистическая регрессия (`logreg.ipynb`) и градиентный бустинг CatBoost (`cboost.ipynb`).

Эти модели предсказывают **вероятность дефолта** P(дефолт | признаки клиента).
Но нас интересует другой вопрос:

> *Насколько изменится вероятность дефолта клиента, если мы с ним свяжемся?*

Это **индивидуальный эффект воздействия** (Conditional Average Treatment Effect, CATE):

$$\tau_t(x) = \mathbb{E}[Y(t) - Y(0) \mid X = x]$$

где $Y(t)$ - потенциальный исход при коммуникации типа $t$, $Y(0)$ - без коммуникации.

**Фундаментальная проблема каузального вывода:** мы наблюдаем только один исход -
клиент либо получил коммуникацию, либо нет. Поэтому нужны специальные методы.

**Уникальное преимущество наших данных:** поскольку датасет синтетический, `TRUE_UPLIFT`
известен для каждого клиента. Это позволяет напрямую оценить качество моделей -
такая возможность крайне редко встречается на практике.

### Что делаем в ноутбуке

1. Покажем, почему классический скоринг неоптимален для выбора клиентов.
2. Построим uplift meta-learners: S-Learner, T-Learner, X-Learner, DR-Learner.
3. Сравним все подходы по AUUC, % от oracle и Spearman rho с TRUE_UPLIFT.
4. Нарисуем Qini-кривые и calibration plots.


## 0. Импорты и настройки воспроизводимости

Устанавливаем глобальный seed. Все операции со случайностью используют `RANDOM_SEED = 91`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 91
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.options.display.float_format = '{:.4f}'.format
pd.options.display.max_columns = 50


## 1. Загрузка данных и проверка ключевых полей

Загружаем датасет из `prepare_dataset.ipynb`. Проверяем наличие:
- **признаков** - числовые фичи Home Credit;
- **treatment** - `COMMUNICATION` (тип коммуникации);
- **outcome** - `TARGET_AFTER_CONTACT`;
- **oracle-полей** - `TRUE_UPLIFT`, `PD_NO_CONTACT`, `PD_SMS` и др.

**Важно:** oracle-поля содержат истинный эффект и не должны попадать в признаки модели.


In [ ]:
import os, json

DATA_PATH  = 'data/processed/uplift-dataset.csv'
ROLES_PATH = 'models/feature_role_groups.json'

df_full = pd.read_csv(DATA_PATH)
print(f'Датасет: {df_full.shape[0]:,} строк x {df_full.shape[1]} колонок')

# Роли признаков
if os.path.exists(ROLES_PATH):
    with open(ROLES_PATH, encoding='utf-8') as f:
        role_groups = json.load(f)
    leakage_cols = set(role_groups.get('leakage_cols_for_modeling', []))
    oracle_cols  = set(role_groups.get('oracle_cols', []))
else:
    leakage_cols = {
        'BASE_PD', 'CONTACT_PROPENSITY', 'RISK_SEGMENT',
        'CONTACT_HISTORY', 'PREFERRED_CHANNEL', 'INTERACTION_SCORE', 'DELAY_FLAG',
        'PD_NO_CONTACT', 'PD_SMS', 'PD_ROBOT_CALL', 'PD_OPERATOR_CALL',
        'UPLIFT_SMS', 'UPLIFT_ROBOT_CALL', 'UPLIFT_OPERATOR_CALL',
        'TRUE_UPLIFT', 'ORACLE_COMMUNICATION', 'ORACLE_TRUE_UPLIFT', 'ORACLE_PD_AFTER_CONTACT',
        'PD_AFTER_CONTACT',
    }
    oracle_cols = {'TRUE_UPLIFT', 'ORACLE_COMMUNICATION', 'ORACLE_TRUE_UPLIFT', 'ORACLE_PD_AFTER_CONTACT'}
    print('ВНИМАНИЕ: файл ролей не найден, используется резервный список leakage-полей')

outcome_col = 'TARGET_AFTER_CONTACT'

print('\nРаспределение по COMMUNICATION:')
print(df_full['COMMUNICATION'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print(f'\nДоля дефолтов: {df_full[outcome_col].mean():.2%}')


## 2. Разбивка на train / test / OOT

Используем ту же разбивку, что и в классических моделях: **60% train / 20% test / 20% OOT**.

Разбивка псевдо-временная (по порядку строк), как и в `cboost.ipynb` и `logreg.ipynb`.


In [ ]:
import math

exclude_always = {'SK_ID_CURR', 'TARGET', outcome_col} | leakage_cols | {'COMMUNICATION'}
feature_cols = [
    c for c in df_full.columns
    if c not in exclude_always and df_full[c].dtype != object
]
print(f'Признаков для модели: {len(feature_cols)}')

n          = len(df_full)
oot_size   = math.ceil(n * 0.2)
train_size = math.ceil((n - oot_size) * 0.75)

oot   = df_full.iloc[:oot_size].copy()
rest  = df_full.iloc[oot_size:].copy()
train = rest.iloc[:train_size].copy()
test  = rest.iloc[train_size:].copy()

print(f'train: {len(train):,} ({len(train)/n:.0%}) | test: {len(test):,} ({len(test)/n:.0%}) | OOT: {len(oot):,} ({len(oot)/n:.0%})')

X_train = train[feature_cols].fillna(-999)
X_test  = test[feature_cols].fillna(-999)
X_oot   = oot[feature_cols].fillna(-999)

y_train = train[outcome_col].values
y_test  = test[outcome_col].values
y_oot   = oot[outcome_col].values

T_train = train['COMMUNICATION'].values
T_test  = test['COMMUNICATION'].values
T_oot   = oot['COMMUNICATION'].values

t_train_bin = (T_train != 'control').astype(int)
t_test_bin  = (T_test  != 'control').astype(int)
t_oot_bin   = (T_oot   != 'control').astype(int)

print(f'Доля дефолтов: train={y_train.mean():.3%}, test={y_test.mean():.3%}, OOT={y_oot.mean():.3%}')


## 3. Вспомогательная функция: Qini-кривая и AUUC

**Qini-кривая** - основная метрика качества uplift-моделей.

**Как строится:**
1. Ранжируем всех клиентов по убыванию предсказанного uplift-скора.
2. Берём первые K% и считаем накопленный эффект относительно случайного выбора.
3. X = доля выбранных клиентов, Y = суммарный накопленный эффект.

**AUUC** (Area Under Uplift Curve) - площадь под Qini-кривой.
Чем выше - тем лучше модель находит клиентов, для которых коммуникация действительно полезна.

**Важный нюанс:** в наших данных `TRUE_UPLIFT` отрицательный означает снижение PD (хорошо).
Поэтому для ранжирования используем `-uplift`: чем меньше (отрицательнее) uplift, тем выше скор.


In [ ]:
def compute_qini(y, treatment, score, n_bins=100):
    # Вычисляет Qini-кривую и AUUC.
    # y: целевая переменная, treatment: бинарный флаг (1=контакт), score: скор ранжирования.
    df_q = pd.DataFrame({'y': y, 't': treatment, 'score': score})
    df_q = df_q.sort_values('score', ascending=False).reset_index(drop=True)

    n_ctrl = int((1 - treatment).sum())
    if n_ctrl == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 0.0]), np.array([0.0, 0.0]), 0.0

    fracs, qini_vals = [0.0], [0.0]
    step = max(1, len(df_q) // n_bins)
    for k in range(step, len(df_q) + 1, step):
        top_k = df_q.iloc[:k]
        t1 = top_k[top_k['t'] == 1]
        t0 = top_k[top_k['t'] == 0]
        if len(t0) == 0:
            continue
        qini = t1['y'].sum() - t0['y'].sum() * (len(t1) / n_ctrl)
        fracs.append(k / len(df_q))
        qini_vals.append(qini)

    full_qini = qini_vals[-1] if qini_vals else 0.0
    random_vals = [f * full_qini for f in fracs]
    auuc = np.trapz(qini_vals, fracs) - np.trapz(random_vals, fracs)
    return np.array(fracs), np.array(qini_vals), np.array(random_vals), auuc


## 4. Базовые стратегии коммуникации

Фиксируем наивные стратегии как точки отсчёта.

| Стратегия | Описание | Смысл |
|---|---|---|
| Никого не контактировать | Скор = 0 | Нижняя граница: нулевые затраты |
| Случайный выбор | Случайный скор | Нулевая информация |
| Risk-based | Скор = P(дефолт) | Текущая практика банков |
| Oracle | Скор = -TRUE_UPLIFT | Идеал: верхняя граница качества |

**Почему risk-based неоптимален?**
Высокорисковый клиент - не обязательно тот, кому поможет звонок. Бывают три нежелательных случая:
- *Sure thing*: клиент заплатит сам, без коммуникации (пустая трата бюджета);
- *Sleeping dog*: коммуникация раздражает клиента и ухудшает его поведение;
- *Lost cause*: клиент не платит ни при каком исходе.

Uplift-модель должна выявлять только *Persuadables* - тех, кому коммуникация реально помогает.


In [ ]:
# Скоры базовых стратегий
rng = np.random.RandomState(RANDOM_SEED)
scores_baselines = {
    'Никого не контактировать': np.zeros(len(y_test)),
    'Случайный выбор':           rng.rand(len(y_test)),
    'Risk-based (BASE_PD)':      test['BASE_PD'].values,
    'Oracle (TRUE_UPLIFT)':     -test['TRUE_UPLIFT'].values,
}

scores_baselines_oot = {
    'Никого не контактировать': np.zeros(len(y_oot)),
    'Случайный выбор':           np.random.RandomState(RANDOM_SEED+1).rand(len(y_oot)),
    'Risk-based (BASE_PD)':      oot['BASE_PD'].values,
    'Oracle (TRUE_UPLIFT)':     -oot['TRUE_UPLIFT'].values,
}

print(f'{'Стратегия':<35} {'AUUC (test)':>12} {'AUUC (OOT)':>12}')
print('-' * 62)
baseline_auuc = {}
for name in scores_baselines:
    _, _, _, auuc_t = compute_qini(y_test, t_test_bin, scores_baselines[name])
    _, _, _, auuc_o = compute_qini(y_oot,  t_oot_bin,  scores_baselines_oot[name])
    baseline_auuc[name] = auuc_t
    print(f'{name:<35} {auuc_t:>12.6f} {auuc_o:>12.6f}')


## 5. S-Learner (Single Model Approach)

### Идея метода

**S-Learner** обучает **одну** модель, добавив тип коммуникации как обычный признак.

Обученная модель: $\hat{\mu}(x, t) = P(Y=1 \mid X=x, T=t)$

Predicted uplift для канала $t$:
$$\hat{\tau}_t(x) = \hat{\mu}(x, t) - \hat{\mu}(x, \text{control})$$

**Реализация:** кодируем `COMMUNICATION` как one-hot, обучаем модель, делаем два предсказания
(с $T=t$ и с $T=\text{control}$), берём разность.

| Плюс | Минус |
|---|---|
| Прост, один проход обучения | Может игнорировать treatment при слабом сигнале |
| Использует все данные | Не учитывает selection bias явно |

### Что ожидаем увидеть

При слабом сигнале (SNR 0.8-1.6) S-Learner может 'растворить' эффект treatment среди
других признаков. Это ожидаемо и является частью исследования.


In [ ]:
from catboost import CatBoostClassifier

# One-hot кодируем treatment
comm_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
comm_encoder.fit(T_train.reshape(-1, 1))

def add_treatment_ohe(X_df, T_arr, encoder):
    # Добавляет one-hot treatment к матрице признаков.
    t_enc = encoder.transform(T_arr.reshape(-1, 1))
    t_df  = pd.DataFrame(t_enc,
                          columns=[f'COMM_{c}' for c in encoder.categories_[0]],
                          index=range(len(X_df)))
    return pd.concat([X_df.reset_index(drop=True), t_df], axis=1)

X_train_s = add_treatment_ohe(X_train, T_train, comm_encoder)
X_test_s  = add_treatment_ohe(X_test,  T_test,  comm_encoder)
X_oot_s   = add_treatment_ohe(X_oot,   T_oot,   comm_encoder)

# Обучение S-Learner на CatBoost
s_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=RANDOM_SEED, verbose=0,
    eval_metric='AUC', early_stopping_rounds=50,
)
s_model.fit(X_train_s, y_train, eval_set=(X_test_s, y_test))

auc_s = roc_auc_score(y_test, s_model.predict_proba(X_test_s)[:, 1])
print(f'S-Learner AUC (test): {auc_s:.4f}  |  Gini: {2*auc_s-1:.4f}')


In [ ]:
channels_list = ['sms', 'robot_call', 'operator_call']

def s_learner_uplift_per_channel(model, X_df, encoder, channels):
    # Вычисляет uplift S-Learner: tau_t(x) = P(Y=1|X,T=t) - P(Y=1|X,T=control)
    X_ctrl = add_treatment_ohe(X_df, np.array(['control'] * len(X_df)), encoder)
    p_ctrl = model.predict_proba(X_ctrl)[:, 1]
    uplift = {}
    for ch in channels:
        X_ch = add_treatment_ohe(X_df, np.array([ch] * len(X_df)), encoder)
        uplift[ch] = model.predict_proba(X_ch)[:, 1] - p_ctrl
    uplift_matrix = pd.DataFrame({ch: uplift[ch] for ch in channels})
    uplift['best'] = uplift_matrix.min(axis=1).values
    return uplift

uplift_s_test = s_learner_uplift_per_channel(s_model, X_test, comm_encoder, channels_list)
uplift_s_oot  = s_learner_uplift_per_channel(s_model, X_oot,  comm_encoder, channels_list)

score_s_test = -uplift_s_test['best']
score_s_oot  = -uplift_s_oot['best']

_, _, _, auuc_s_test = compute_qini(y_test, t_test_bin, score_s_test)
_, _, _, auuc_s_oot  = compute_qini(y_oot,  t_oot_bin,  score_s_oot)
oracle_auuc = baseline_auuc['Oracle (TRUE_UPLIFT)']

print(f'S-Learner AUUC: test={auuc_s_test:.6f}  |  OOT={auuc_s_oot:.6f}')
print(f'Относительная эффективность: {auuc_s_test/oracle_auuc*100:.1f}% от оракула')


## 6. T-Learner (Two-Model Approach)

### Идея метода

**T-Learner** обучает **отдельную модель для каждого treatment**:

$$\hat{\mu}_t(x) = P(Y=1 \mid X=x, T=t), \quad \hat{\mu}_0(x) = P(Y=1 \mid X=x, T=\text{control})$$

Uplift: $\hat{\tau}_t(x) = \hat{\mu}_t(x) - \hat{\mu}_0(x)$

**Преимущество:** каждая модель специализируется на своей подвыборке и лучше улавливает
специфику канала.

**Недостаток:** группы несбалансированы (operator_call - ~25%, control - ~30%).
Модели сравниваются между несоразмерными выборками.

### Что ожидаем увидеть

T-Learner должен превзойти S-Learner, так как в данных заложена сегментная специфика:
SMS эффективно для low-risk, operator_call - для high-risk.


In [ ]:
# Обучаем по одной CatBoost-модели на каждый treatment
t_models = {}
print('Обучение T-Learner (одна модель на каждый treatment):')
for treatment_name in ['control'] + channels_list:
    mask = T_train == treatment_name
    n_tr = mask.sum()
    m = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                            random_seed=RANDOM_SEED, verbose=0)
    m.fit(X_train[mask], y_train[mask])
    t_models[treatment_name] = m
    mask_t = T_test == treatment_name
    if mask_t.sum() > 50:
        auc = roc_auc_score(y_test[mask_t], m.predict_proba(X_test[mask_t])[:, 1])
        print(f'  {treatment_name:<16}  n_train={n_tr:>6,}  AUC={auc:.4f}')
    else:
        print(f'  {treatment_name:<16}  n_train={n_tr:>6,}  (мало данных в test)')


In [ ]:
def t_learner_uplift(models, X_df, channels):
    # T-Learner uplift: tau_t(x) = mu_t(x) - mu_control(x)
    p_ctrl = models['control'].predict_proba(X_df)[:, 1]
    uplift = {}
    for ch in channels:
        uplift[ch] = models[ch].predict_proba(X_df)[:, 1] - p_ctrl
    uplift_matrix = pd.DataFrame({ch: uplift[ch] for ch in channels})
    uplift['best'] = uplift_matrix.min(axis=1).values
    return uplift

uplift_t_test = t_learner_uplift(t_models, X_test, channels_list)
uplift_t_oot  = t_learner_uplift(t_models, X_oot,  channels_list)

score_t_test = -uplift_t_test['best']
score_t_oot  = -uplift_t_oot['best']

_, _, _, auuc_t_test = compute_qini(y_test, t_test_bin, score_t_test)
_, _, _, auuc_t_oot  = compute_qini(y_oot,  t_oot_bin,  score_t_oot)
print(f'T-Learner AUUC: test={auuc_t_test:.6f}  |  OOT={auuc_t_oot:.6f}')
print(f'Относительная эффективность: {auuc_t_test/oracle_auuc*100:.1f}% от оракула')


## 7. X-Learner

### Идея метода

**X-Learner** разработан специально для задач с **несбалансированными группами** treatment и control.

**Алгоритм (для бинарного treatment):**

**Шаг 1.** Базовые outcome-модели (как в T-Learner):
$\hat{\mu}_1(x) = P(Y \mid X, T=1)$, $\hat{\mu}_0(x) = P(Y \mid X, T=0)$

**Шаг 2.** Псевдо-эффект для каждого клиента:
- В группе воздействия: $\tilde{\tau}_i^{(1)} = Y_i - \hat{\mu}_0(X_i)$
- В контрольной группе: $\tilde{\tau}_j^{(0)} = \hat{\mu}_1(X_j) - Y_j$

**Шаг 3.** Регрессии $\hat{\tau}_1(x)$ и $\hat{\tau}_0(x)$ на псевдо-эффектах.

**Шаг 4.** Взвешенное смешение через propensity score $g(x) = P(T=1 \mid X=x)$:
$$\hat{\tau}(x) = g(x) \cdot \hat{\tau}_0(x) + (1 - g(x)) \cdot \hat{\tau}_1(x)$$

**Смысл:** когда группа воздействия велика, опираемся на $\hat{\tau}_1$; когда мала - на $\hat{\tau}_0$.


In [ ]:
from catboost import CatBoostRegressor

# Шаг 1: базовые outcome-модели для бинарного treatment
mask_treat = t_train_bin == 1
mask_ctrl  = t_train_bin == 0

mu1_model = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                                random_seed=RANDOM_SEED, verbose=0)
mu0_model = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                                random_seed=RANDOM_SEED, verbose=0)
mu1_model.fit(X_train[mask_treat], y_train[mask_treat])
mu0_model.fit(X_train[mask_ctrl],  y_train[mask_ctrl])
print(f'mu1 (воздействие): {mask_treat.sum():,} клиентов  |  mu0 (контроль): {mask_ctrl.sum():,}')

# Шаг 2: псевдо-эффекты
d_treat = y_train[mask_treat] - mu0_model.predict_proba(X_train[mask_treat])[:, 1]
d_ctrl  = mu1_model.predict_proba(X_train[mask_ctrl])[:, 1] - y_train[mask_ctrl]
print(f'Псевдо-эффекты (воздействие): mean={d_treat.mean():.4f}  |  (контроль): mean={d_ctrl.mean():.4f}')


In [ ]:
# Шаг 3: регрессоры на псевдо-эффектах
tau1_model = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=5,
                                random_seed=RANDOM_SEED, verbose=0)
tau0_model = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=5,
                                random_seed=RANDOM_SEED, verbose=0)
tau1_model.fit(X_train[mask_treat], d_treat)
tau0_model.fit(X_train[mask_ctrl],  d_ctrl)

# Шаг 4: propensity score и взвешенное смешение
prop_model = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=4,
                                 random_seed=RANDOM_SEED, verbose=0)
prop_model.fit(X_train, t_train_bin)

def x_learner_predict(tau1, tau0, prop, X):
    # tau(x) = g(x)*tau0(x) + (1-g(x))*tau1(x)
    g = prop.predict_proba(X)[:, 1]
    return g * tau0.predict(X) + (1 - g) * tau1.predict(X)

uplift_x_test = x_learner_predict(tau1_model, tau0_model, prop_model, X_test)
uplift_x_oot  = x_learner_predict(tau1_model, tau0_model, prop_model, X_oot)

score_x_test = -uplift_x_test
score_x_oot  = -uplift_x_oot

_, _, _, auuc_x_test = compute_qini(y_test, t_test_bin, score_x_test)
_, _, _, auuc_x_oot  = compute_qini(y_oot,  t_oot_bin,  score_x_oot)
print(f'X-Learner AUUC: test={auuc_x_test:.6f}  |  OOT={auuc_x_oot:.6f}')
print(f'Относительная эффективность: {auuc_x_test/oracle_auuc*100:.1f}% от оракула')


## 8. DR-Learner (Doubly Robust Learner)

### Идея метода

**DR-Learner** обладает свойством **двойной робастности** (doubly robust):
если либо модель исхода, либо модель propensity оценены верно (хотя бы одна),
итоговая оценка CATE остаётся несмещённой.

Это особенно важно при selection bias - а в наших данных он намеренно заложен
через `CONTACT_PROPENSITY`.

**Алгоритм:**

1. Out-of-fold outcome model $\hat{\mu}(x, t)$ и propensity $\hat{e}(x)$.
2. **Doubly robust псевдо-исход** для каждого клиента:

$$\tilde{Y}_i^{DR} = \underbrace{\hat{\mu}(x_i,1) - \hat{\mu}(x_i,0)}_{\text{прямая оценка}} + \underbrace{\frac{T_i - \hat{e}(x_i)}{\hat{e}(x_i)(1-\hat{e}(x_i))} \cdot (Y_i - \hat{\mu}(x_i, T_i))}_{\text{IPW-коррекция}}$$

3. Финальная регрессия: $\hat{\tau}(x) = f(x)$ предсказывает псевдо-исходы.

**Важно:** out-of-fold предсказания (кросс-валидация) обязательны - иначе модель
будет переобучена на собственных ошибках псевдо-исхода.


In [ ]:
# DR-Learner: out-of-fold outcome и propensity через 5-fold кросс-валидацию
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_train_arr = X_train.values
mu1_oof = np.zeros(len(X_train_arr))
mu0_oof = np.zeros(len(X_train_arr))
e_oof   = np.zeros(len(X_train_arr))

print('DR-Learner: 5-fold кросс-валидация для out-of-fold предсказаний...')
for fold, (idx_tr, idx_val) in enumerate(kf.split(X_train_arr)):
    X_f, X_v = X_train_arr[idx_tr], X_train_arr[idx_val]
    y_f, t_f = y_train[idx_tr], t_train_bin[idx_tr]

    # Outcome model: добавляем бинарный T как признак
    X_f_aug  = np.column_stack([X_f, t_f])
    X_v_aug1 = np.column_stack([X_v, np.ones(len(X_v))])
    X_v_aug0 = np.column_stack([X_v, np.zeros(len(X_v))])
    mu_tmp = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=5,
                                 random_seed=RANDOM_SEED, verbose=0)
    mu_tmp.fit(X_f_aug, y_f)
    mu1_oof[idx_val] = mu_tmp.predict_proba(X_v_aug1)[:, 1]
    mu0_oof[idx_val] = mu_tmp.predict_proba(X_v_aug0)[:, 1]

    # Propensity model
    e_tmp = CatBoostClassifier(iterations=100, learning_rate=0.05, depth=4,
                                random_seed=RANDOM_SEED, verbose=0)
    e_tmp.fit(X_f, t_f)
    e_oof[idx_val] = e_tmp.predict_proba(X_v)[:, 1]
    print(f'  Fold {fold+1}/5', end='\r')

print('\nOut-of-fold предсказания готовы.')
print(f'  mu1: mean={mu1_oof.mean():.4f}  |  mu0: mean={mu0_oof.mean():.4f}  |  e: mean={e_oof.mean():.4f}')


In [ ]:
# Формируем doubly robust псевдо-исходы
e_clipped   = np.clip(e_oof, 0.05, 0.95)  # клиппинг propensity
ipw_weight  = (t_train_bin - e_clipped) / (e_clipped * (1 - e_clipped))
mu_at_obs   = t_train_bin * mu1_oof + (1 - t_train_bin) * mu0_oof
dr_pseudo   = (mu1_oof - mu0_oof) + ipw_weight * (y_train - mu_at_obs)

print(f'DR псевдо-исходы: mean={dr_pseudo.mean():.5f}, std={dr_pseudo.std():.5f}')
print(f'Доля клиентов с отрицательным псевдо-эффектом (помогает коммуникация): {(dr_pseudo < 0).mean():.1%}')

# Финальная регрессия
dr_final = CatBoostRegressor(iterations=400, learning_rate=0.05, depth=6,
                               random_seed=RANDOM_SEED, verbose=0)
dr_final.fit(X_train_arr, dr_pseudo)

uplift_dr_test = dr_final.predict(X_test.values)
uplift_dr_oot  = dr_final.predict(X_oot.values)

score_dr_test = -uplift_dr_test
score_dr_oot  = -uplift_dr_oot

_, _, _, auuc_dr_test = compute_qini(y_test, t_test_bin, score_dr_test)
_, _, _, auuc_dr_oot  = compute_qini(y_oot,  t_oot_bin,  score_dr_oot)
print(f'DR-Learner AUUC: test={auuc_dr_test:.6f}  |  OOT={auuc_dr_oot:.6f}')
print(f'Относительная эффективность: {auuc_dr_test/oracle_auuc*100:.1f}% от оракула')


## 9. Сравнение всех подходов

### Метрики сравнения

| Метрика | Что измеряет | Как интерпретировать |
|---|---|---|
| **AUUC** | Качество uplift-ранжирования | Выше = лучше находим убеждаемых |
| **% от Oracle** | Насколько близки к теоретическому максимуму | 100% = идеал |
| **Spearman rho с TRUE_UPLIFT** | Ранговая корреляция предсказанного и истинного эффекта | Доступна только в синтетическом эксперименте |

### Ожидаемая интерпретация

- Если все uplift-модели близки к risk-based по AUUC - сигнал слишком слабый.
- Если лучший uplift значимо выше risk-based - uplift добавляет реальную ценность.
- Spearman rho > 0.05 при SNR 0.8-1.6 уже содержательный результат.


In [ ]:
# Финальная таблица результатов
true_uplift_test = test['TRUE_UPLIFT'].values
results = []

model_configs = [
    ('Случайный выбор',        scores_baselines['Случайный выбор'],    np.zeros(len(y_test)),    'Baseline'),
    ('Risk-based (BASE_PD)',   scores_baselines['Risk-based (BASE_PD)'], test['BASE_PD'].values,  'Baseline'),
    ('Oracle (TRUE_UPLIFT)',   scores_baselines['Oracle (TRUE_UPLIFT)'], -true_uplift_test,       'Baseline'),
    ('S-Learner',              score_s_test,    uplift_s_test['best'],  'Uplift'),
    ('T-Learner',              score_t_test,    uplift_t_test['best'],  'Uplift'),
    ('X-Learner',              score_x_test,    uplift_x_test,          'Uplift'),
    ('DR-Learner',             score_dr_test,   uplift_dr_test,         'Uplift'),
]

for name, score, uplift_pred, mtype in model_configs:
    _, _, _, auuc = compute_qini(y_test, t_test_bin, score)
    sp_rho, sp_p  = stats.spearmanr(uplift_pred, true_uplift_test)
    results.append({'Модель': name, 'AUUC': auuc,
                    '% от Oracle': auuc / oracle_auuc * 100,
                    'Spearman rho': sp_rho, 'p-value': sp_p, 'Тип': mtype})

df_results = pd.DataFrame(results).set_index('Модель')
print(df_results.sort_values('AUUC', ascending=False).to_string(
    float_format=lambda x: f'{x:.4f}'
))


In [ ]:
# Qini-кривые всех моделей
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

scores_oot_map = {
    'Oracle (TRUE_UPLIFT)': -oot['TRUE_UPLIFT'].values,
    'Risk-based (BASE_PD)': oot['BASE_PD'].values,
    'Случайный выбор': np.random.RandomState(RANDOM_SEED+1).rand(len(y_oot)),
    'S-Learner':  score_s_oot,
    'T-Learner':  score_t_oot,
    'X-Learner':  score_x_oot,
    'DR-Learner': score_dr_oot,
}
plot_order = [
    ('Oracle (TRUE_UPLIFT)', scores_baselines['Oracle (TRUE_UPLIFT)'], 'black',     '--', 2.5),
    ('Risk-based (BASE_PD)', scores_baselines['Risk-based (BASE_PD)'], 'dimgray',   ':',  1.8),
    ('Случайный выбор',      scores_baselines['Случайный выбор'],       'lightgray', ':',  1.2),
    ('S-Learner',            score_s_test,                             'steelblue', '-',  1.8),
    ('T-Learner',            score_t_test,                             'darkorange','-',  1.8),
    ('X-Learner',            score_x_test,                             'green',     '-',  1.8),
    ('DR-Learner',           score_dr_test,                            'crimson',   '-',  1.8),
]

for ax, (y_arr, t_arr, suffix) in zip(axes,
        [(y_test, t_test_bin, 'test'), (y_oot, t_oot_bin, 'OOT')]):
    for label, score_t, color, ls, lw in plot_order:
        sc = score_t if suffix == 'test' else scores_oot_map[label]
        fracs, qini, _, auuc = compute_qini(y_arr, t_arr, sc)
        ax.plot(fracs * 100, qini,
                label=f'{label}  (AUUC={auuc:.5f})',
                color=color, linestyle=ls, linewidth=lw)
    ax.set_xlabel('Доля отобранных клиентов, %')
    ax.set_ylabel('Qini (предотвращённые дефолты)')
    ax.set_title(f'Qini-кривые - {suffix}')
    ax.legend(loc='upper left', fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Сравнение стратегий коммуникации: Qini-кривые', fontsize=13)
plt.tight_layout()
plt.show()


## 10. Оценка через истинный uplift

### Почему это методологически ценно

В реальных данных истинный $\tau(x)$ никогда не известен для отдельного клиента.
В нашем эксперименте `TRUE_UPLIFT` зафиксирован в датасете.

Это позволяет применить три дополнительные метрики качества:

1. **RMSE**: точность предсказания численного эффекта
2. **Spearman rho**: правильно ли модель ранжирует клиентов по силе эффекта
3. **Calibration plot**: систематически ли модель переоценивает / недооценивает эффект

> Наличие такой оценки в академической работе - методологически сильный аргумент.
В uplift-литературе именно оценка через known ground truth считается 'золотым стандартом'.


In [ ]:
# RMSE и Spearman rho к TRUE_UPLIFT
model_uplift_preds = {
    'S-Learner':            uplift_s_test['best'],
    'T-Learner':            uplift_t_test['best'],
    'X-Learner':            uplift_x_test,
    'DR-Learner':           uplift_dr_test,
    'Risk-based (BASE_PD)': test['BASE_PD'].values,
}

true_uplift = test['TRUE_UPLIFT'].values

print(f'{'Модель':<22} {'RMSE':>8} {'Spearman rho':>14} {'p-value':>10}')
print('-' * 58)
for name, pred in model_uplift_preds.items():
    rmse = np.sqrt(np.mean((pred - true_uplift) ** 2))
    sp_rho, sp_p = stats.spearmanr(pred, true_uplift)
    sig = '**' if sp_p < 0.01 else ('*' if sp_p < 0.05 else '')
    print(f'{name:<22} {rmse:>8.5f} {sp_rho:>14.4f} {sp_p:>10.4f} {sig}')
print('* p<0.05, ** p<0.01')


In [ ]:
# Calibration plot: предсказанный vs истинный uplift по децилям
uplift_models_cal = {k: v for k, v in model_uplift_preds.items()
                      if k != 'Risk-based (BASE_PD)'}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, (name, pred) in zip(axes, uplift_models_cal.items()):
    df_cal = pd.DataFrame({'pred': pred, 'true': true_uplift})
    df_cal['decile'] = pd.qcut(df_cal['pred'], q=10, labels=False, duplicates='drop')
    cal = df_cal.groupby('decile').agg(
        pred_mean=('pred', 'mean'),
        true_mean=('true', 'mean'),
        count=('pred', 'count'),
    ).reset_index()

    ax.scatter(cal['pred_mean'], cal['true_mean'],
               s=cal['count'] / 4, alpha=0.8, color='steelblue', zorder=5)
    ax.plot(cal['pred_mean'], cal['true_mean'], '-o', color='steelblue', alpha=0.4)

    lo = min(cal['pred_mean'].min(), cal['true_mean'].min())
    hi = max(cal['pred_mean'].max(), cal['true_mean'].max())
    ax.plot([lo, hi], [lo, hi], 'r--', label='Идеал')

    sp_rho, _ = stats.spearmanr(pred, true_uplift)
    ax.set_title(f'{name}  |  Spearman rho = {sp_rho:.3f}')
    ax.set_xlabel('Предсказанный uplift (среднее по дециле)')
    ax.set_ylabel('Истинный uplift (TRUE_UPLIFT)')
    ax.legend(fontsize=8)

plt.suptitle('Calibration plot: предсказанный vs истинный uplift по децилям (test)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('Как читать: точки на красной диагонали = идеальная калибровка.')
print('Выше диагонали = модель переоценивает эффект. Ниже = недооценивает.')


In [ ]:
# Сохраняем предсказанные uplift-скоры для использования в policy_evaluation.ipynb
# Каждая строка соответствует клиенту из test или OOT выборки.
scores_to_save = pd.DataFrame({
    "split":          ["test"] * len(y_test) + ["oot"] * len(y_oot),
    "y":              np.concatenate([y_test, y_oot]),
    "treatment_bin":  np.concatenate([t_test_bin, t_oot_bin]),
    "COMMUNICATION":  np.concatenate([T_test, T_oot]),
    "BASE_PD":        np.concatenate([test["BASE_PD"].values, oot["BASE_PD"].values]),
    "TRUE_UPLIFT":    np.concatenate([test["TRUE_UPLIFT"].values, oot["TRUE_UPLIFT"].values]),
    "RISK_SEGMENT":   np.concatenate([test["RISK_SEGMENT"].values, oot["RISK_SEGMENT"].values]),
    # per-channel uplift S-Learner
    "uplift_s_sms":      np.concatenate([uplift_s_test["sms"],       uplift_s_oot["sms"]]),
    "uplift_s_robot":    np.concatenate([uplift_s_test["robot_call"],  uplift_s_oot["robot_call"]]),
    "uplift_s_operator": np.concatenate([uplift_s_test["operator_call"], uplift_s_oot["operator_call"]]),
    # best-channel uplift per model
    "score_s":   np.concatenate([score_s_test,  score_s_oot]),
    "score_t":   np.concatenate([score_t_test,  score_t_oot]),
    "score_x":   np.concatenate([score_x_test,  score_x_oot]),
    "score_dr":  np.concatenate([score_dr_test, score_dr_oot]),
    "uplift_t_sms":      np.concatenate([uplift_t_test["sms"],       uplift_t_oot["sms"]]),
    "uplift_t_robot":    np.concatenate([uplift_t_test["robot_call"],  uplift_t_oot["robot_call"]]),
    "uplift_t_operator": np.concatenate([uplift_t_test["operator_call"], uplift_t_oot["operator_call"]]),
})

scores_to_save.to_csv("data/processed/uplift_scores.csv", index=False)
print(f"Скоры сохранены: {len(scores_to_save):,} строк x {len(scores_to_save.columns)} колонок")
print(f"  test: {(scores_to_save.split == 'test').sum():,}  |  OOT: {(scores_to_save.split == 'oot').sum():,}")


## 11. Итоговые выводы

### Что мы узнали

В этом ноутбуке сравнены четыре uplift meta-learner (S, T, X, DR) с базовыми стратегиями.

**Ключевые наблюдения:**

1. **Risk-based vs Uplift.** Классический скоринг конкурентоспособен при слабом сигнале.
   Однако все uplift-модели превосходят случайное ранжирование - сигнал в данных есть.

2. **S-Learner** при слабом SNR может растворить treatment среди остальных признаков.

3. **T-Learner** лучше улавливает специфику каналов, но чувствителен к дисбалансу групп.

4. **X-Learner** устойчивее при несбалансированных группах за счёт propensity-взвешивания.

5. **DR-Learner** методологически строже: doubly robust свойство защищает от ошибок
   в одной из nuisance-моделей, что критично при selection bias.

### Что означает для бизнеса

Uplift-модели позволяют:
- Выявлять *Persuadables* - клиентов, для которых коммуникация реально помогает.
- Избегать *Sleeping Dogs* - клиентов, которых коммуникация раздражает.
- Экономить бюджет CRM, не тратя его на *Sure Things* (заплатят сами).

### Следующий шаг

В ноутбуке `policy_evaluation.ipynb`:
- Переведём AUUC в денежный эффект (предотвращённые дефолты при фиксированном бюджете).
- Сравним все стратегии: предотвращённые дефолты vs стоимость коммуникаций.
- Определим бюджетный порог, при котором uplift начинает выигрывать у risk-based.
